# Card-Interpretation Evals

Runs the **production agent harness** (`agent.runtime.run_agent`) over a benchmark and measures: tool calls, answer similarity to the human canonical, executability, the *did-something* (no-op) rate, cost, and latency.

The harness reproduces real play: a live parity `GameState`, an actor who is **not** the card's author, the production toolbox (optionally filtered), the honored caps, and vision when enabled.

**Setup:** `uv sync --group evals`, select this repo's venv as the kernel, and configure the LLM gateway (`LLM_API_KEY` / `LLM_BASE_URL`) — the agent and the judge both call it.

Workflow: **1) Configure → 2) Run (auto-persists) → 3) Load history → 4) Compare → 5) Drill down.**

## Metric definitions

**LLM-judge scorers** (0–1, scored by a second model reading the card text and the generated effect; only when `use_judge=True`):

- `intent_match` — does the generated effect do what the card text asks?
- `target_accuracy` — does the effect hit the right target (self vs a chosen player vs everyone)?
- `persistence_accuracy` — is a one-shot effect one-shot, and an ongoing rule an ongoing rule with the right trigger?
- `magnitude_sign` — is the direction of the effect right (helps who it should help, hurts who it should hurt)?

**Deterministic scorers** (no LLM involved):

- `dsl_validity` — the plan is non-empty and every snippet/hook passes static sandbox validation.
- `executability` — the emitted plan compiles and dry-runs without error. Note: it doesn't look at the verdict, so even the agent's error-fallback plan (a single note op) passes — by design; see the callout below.
- `did_something` — the verdict isn't `invalid` AND the dry-run is clean AND at least one *mechanical* (non-note) op is emitted. This is the "did the card actually affect the game" check.
- `sandbox_behavior` — similarity of the ops the plan produces vs the expected ops on canned fixtures (1.0 when a card has no expected sandbox).

**Run-level aggregates** (in the summary table):

- `invalid_rate` — fraction of cards where the final verdict was `invalid`, whether the agent judged the card uninterpretable or fell back after an error.
- `agent_error_rate` — fraction of cards where the agent *crashed, timed out, or failed to parse its own answer* and returned the bounded fallback. High values mean the run is measuring infrastructure failures, not card interpretation.
- `consistency` — only when `n_samples > 1`: within-card score stability across samples (lower stdev = more deterministic agent).
- Plus cost/latency/token/tool-call stats (`mean_cost_usd`, `p50_latency_ms`, `mean_tool_calls`, …).

> **Reading divergence:** high `executability` with low `did_something` (or high `invalid_rate` / `agent_error_rate`) does **not** mean good plans — it means the agent is erroring into fallback no-op plans that trivially "execute". Check the run log for gateway errors (e.g. HTTP 402 `budget_exceeded`) before trusting any other metric.

In [ ]:
import logging
import sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # make src/ importable from scripts/

import matplotlib.pyplot as plt

from config import EVAL_BENCHMARKS, EVAL_MODEL_PRICES
from evals import store, viz
from evals.runner import EvalConfig, available_tool_names, run_benchmark

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("evals.runner").setLevel(logging.INFO)

## 1. Configure a run

Every knob on `EvalConfig`:

| Knob | Meaning |
|---|---|
| `benchmark` | `seed` / `eval` / `eval_hard` / `real` |
| `model_name` | chat model id; `None` = configured `LLM_CHAT_MODEL` |
| `enabled_tools` | `None` = full production toolbox; a `frozenset` of names from `TOOLS` below; `frozenset()` = no tools |
| `max_tool_calls` | agent step cap (production default 24); `None` = default |
| `timeout` | wall-clock seconds per card; `None` = default (600) |
| `vision` | attach card art / alt_text as an image (needs a vision model) |
| `n_samples` | samples per card; `>1` adds a consistency (stochastic reliability) metric |
| `sample_size` | cap cards per benchmark; `None` = all (**set one for `real`** — 698 cards) |
| `concurrency` | cards run in parallel threads; 4–8 is a good speedup for big runs |
| `use_judge` | run the LLM judge (costs money) vs deterministic scorers only |
| `tracing` | LangSmith tracing during the run; `False` (default) suppresses it to save costs, `True` inherits the ambient `LANGSMITH_TRACING` setting |
| `label` | short name shown in charts |

In [ ]:
TOOLS = available_tool_names()

print("Benchmarks: ", ", ".join(f"{name} ({spec['path']})" for name, spec in EVAL_BENCHMARKS.items()))
print("Tools:      ", ", ".join(TOOLS))
print("Prices ($/1M tokens):", EVAL_MODEL_PRICES)

In [ ]:
config = EvalConfig(
    benchmark="seed",
    model_name=None,  # e.g. "us.anthropic.claude-haiku-4-5-20251001-v1:0" or "openai.gpt-5.4"
    enabled_tools=None,  # e.g. frozenset(TOOLS) - {"web_search"}, or frozenset({"card_rag", "dry_run_effect"})
    max_tool_calls=None,  # e.g. 12
    timeout=None,
    vision=False,
    n_samples=1,
    sample_size=None,  # small while iterating; None for the full set
    concurrency=5,  # bump to 4-8 for full-benchmark runs
    use_judge=True,
    tracing=False,  # LangSmith tracing for debugging (off by default for evals)
    label="baseline",
)

# Price overrides (USD per 1M tokens) if your model is missing from EVAL_MODEL_PRICES:
#   from dataclasses import replace
#   config = replace(config, prices={"my-model": {"input": 0.25, "output": 2.0}})
config

## 2. Run it

Expect roughly `cards x n_samples x per-card latency / concurrency`. The run is written to `data/eval/runs/<timestamp>_<benchmark>_<model>.json`, so it survives kernel restarts and can be compared later.

In [ ]:
run = run_benchmark(config, timestamp=datetime.now().strftime("%Y%m%d-%H%M%S"))
print("saved:", store.save_run(run))
viz.summary_table([run.aggregate()])

## 3. Load run history

Loads **every** persisted run in chronological order; slice `payloads` (or pass explicit paths) to compare a subset.

In [ ]:
payloads = store.load_runs()  # or store.load_runs([Path("data/eval/runs/....json"), ...])
summaries = [p["summary"] for p in payloads]
print(f"{len(summaries)} run(s) loaded")
viz.summary_table(summaries)

## 4. Compare runs

Each run keeps one color across every chart. Higher is better for the quality metrics; lower is better for tool calls, cost, and latency.

In [ ]:
viz.plot_quality(summaries)
plt.tight_layout()
plt.show()

In [ ]:
viz.plot_efficiency(summaries)
plt.tight_layout()
plt.show()

In [ ]:
viz.plot_tool_usage(summaries)
plt.tight_layout()
plt.show()

In [ ]:
viz.plot_cost_vs_quality(summaries, quality_key="intent_match")
plt.tight_layout()
plt.show()

## 5. Drill down: worst cards

The lowest-scoring cards for a chosen metric in one run, with the failure reason — where to look when a metric drops. Try `metric="intent_match"`, `"executability"`, `"sandbox_behavior"`, or `"did_something"`.

In [ ]:
viz.worst_cards(payloads[-1], metric="executability", n=10)